# Toxic Comment Classification Using Deep Learning

**AIP301 Hands-on Assignment**

This notebook builds a deep learning model that classifies online comments into toxicity categories.
It is organised under the five assignment tasks:

1. Dataset Loading and Initial Exploration
2. Text Cleaning, Preprocessing, Data Quality, and Target Analysis
3. NLP Tokenization and Sequence Padding
4. Deep Learning Model Development, Training, and Validation
5. Model Evaluation, Visualization, and Prediction

---
## Task 1: Dataset Loading and Initial Exploration

Before cleaning text or building a model, we need to understand the data: what each record looks like,
how many records and columns there are, what type each column has, and whether there are missing or
duplicate values.

### 1.1 Import libraries

- **pandas** loads and explores tabular data (a `DataFrame` is like a spreadsheet in Python).
- **numpy** provides fast numerical arrays.
- **pathlib.Path** builds file paths that work on Windows, macOS, and Linux.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Show all columns and longer text when displaying DataFrames
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

print("pandas version:", pd.__version__)
print("numpy version :", np.__version__)

pandas version: 3.0.6
numpy version : 2.5.3


### 1.2 Locate the project folders

The notebook lives in `notebooks/`, while the dataset lives in `data/`.
This cell finds the project root so the notebook works whether Jupyter is started
from the project root or from the `notebooks/` folder.

In [2]:
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "toxic_comments_dataset.csv"

print("Project root:", PROJECT_ROOT.resolve())
print("Dataset path:", DATA_PATH.resolve())
print("Dataset found:", DATA_PATH.exists())

Project root: D:\CProjects\toxic-comment-classification-dl
Dataset path: D:\CProjects\toxic-comment-classification-dl\data\toxic_comments_dataset.csv
Dataset found: True


### 1.3 Load the dataset

`pd.read_csv()` reads the CSV file into a DataFrame. Each row is one comment,
and each column is one attribute of that comment.

In [3]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Place toxic_comments_dataset.csv inside the data/ folder."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset loaded successfully.")

Dataset loaded successfully.


### 1.4 Display the first 10 records

`df.head(10)` shows the first 10 rows so we can see what real records look like.

In [4]:
df.head(10)

,Comment_ID,Comment_Text,Toxicity_Label,Clean,Toxic,Severe_Toxic,Threat,Identity_Attack,Toxicity_Score,Comment_Length,Word_Count,Platform,Topic,Moderation_Priority,Language
0,COM000001,"In this discussion, the message expresses a clear opinion about the subject. Source wording: Comment for review: ""Th...",Identity_Attack,0,1,0,0,1,0.818,66,10,Community Chat,Sports,High,English
1,COM000002,"Regarding the point raised, the message expresses a clear opinion about the subject. Source wording: Comment for rev...",Clean,1,0,0,0,0,0.069,58,9,News Comments,Entertainment,High,English
2,COM000003,"From the information provided, the message expresses a clear opinion about the subject. Source wording: Comment for ...",Toxic,0,1,0,0,0,0.622,44,7,Social Media,Politics,Low,English
3,COM000004,"In the current conversation, the message expresses a clear opinion about the subject. Source wording: Comment for re...",Clean,1,0,0,0,0,0.010,61,9,Social Media,Entertainment,Low,English
4,COM000005,"Looking at this issue, the message expresses a clear opinion about the subject. Source wording: Comment for review: ...",Clean,1,0,0,0,0,0.140,52,8,Blog,Sports,Low,English
5,COM000006,"For this particular topic, the message expresses a clear opinion about the subject. Source wording: Comment for revi...",Toxic,0,1,0,0,0,0.729,44,7,Social Media,Technology,Low,English
6,COM000007,"In response to the statement, the message expresses a clear opinion about the subject. Source wording: Comment for r...",Clean,1,0,0,0,0,0.193,52,8,Blog,Entertainment,Low,English
7,COM000008,"Considering the context, the message expresses a clear opinion about the subject. Source wording: Comment for review...",Clean,1,0,0,0,0,0.183,61,9,Forum,Gaming,Low,English
8,COM000009,"On this subject, the message expresses a clear opinion about the subject. Source wording: Comment for review: ""This ...",Clean,1,0,0,0,0,0.142,60,8,Social Media,Education,Medium,English
9,COM000010,"With respect to the comment, the message expresses a clear opinion about the subject. Source wording: Comment for re...",Clean,1,0,0,0,0,0.153,61,9,Forum,Entertainment,Low,English


### 1.5 Display the last 8 records

`df.tail(8)` shows the last 8 rows. Checking the end of the file confirms the whole file
was read correctly (for example, no broken rows at the end).

In [5]:
df.tail(8)

,Comment_ID,Comment_Text,Toxicity_Label,Clean,Toxic,Severe_Toxic,Threat,Identity_Attack,Toxicity_Score,Comment_Length,Word_Count,Platform,Topic,Moderation_Priority,Language
29992,COM029993,"Considering the context, the statement addresses a response to the preceding point. Source wording: Comment for revi...",Toxic,0,1,0,0,0,0.520,52,7,Forum,Gaming,Medium,English
29993,COM029994,"On this subject, the statement addresses a response to the preceding point. Source wording: Comment for review: ""I a...",Clean,1,0,0,0,0,0.195,61,9,News Comments,Sports,Medium,English
29994,COM029995,"With respect to the comment, the statement addresses a response to the preceding point. Source wording: Comment for ...",Clean,1,0,0,0,0,0.174,58,9,Community Chat,Gaming,Low,English
29995,COM029996,"In reviewing this exchange, the statement addresses a response to the preceding point. Source wording: Comment for r...",Clean,1,0,0,0,0,0.013,60,8,Community Chat,Sports,Medium,English
29996,COM029997,"For the matter being discussed, the statement addresses a response to the preceding point. Source wording: Comment f...",Clean,1,0,0,0,0,0.050,52,8,Blog,Politics,Medium,English
29997,COM029998,"Based on the message, the statement addresses a response to the preceding point. Source wording: Comment for review:...",Clean,1,0,0,0,0,0.078,61,9,Forum,Sports,Low,English
29998,COM029999,"In the wider discussion, the statement addresses a response to the preceding point. Source wording: Comment for revi...",Clean,1,0,0,0,0,0.003,60,8,Social Media,Sports,Medium,English
29999,COM030000,"As part of this conversation, the statement addresses a response to the preceding point. Source wording: Comment for...",Clean,1,0,0,0,0,0.082,61,9,Community Chat,Education,High,English


### 1.6 Dataset dimensions

`df.shape` returns a tuple `(number_of_rows, number_of_columns)`.

In [6]:
n_rows, n_cols = df.shape
print(f"Number of rows (comments): {n_rows:,}")
print(f"Number of columns        : {n_cols}")

Number of rows (comments): 30,000
Number of columns        : 15


### 1.7 Column names

`df.columns` lists every column name. We use these real names later to choose the
text column and the target column (we do not assume them).

In [7]:
for i, col in enumerate(df.columns, start=1):
    print(f"{i:2d}. {col}")

 1. Comment_ID
 2. Comment_Text
 3. Toxicity_Label
 4. Clean
 5. Toxic
 6. Severe_Toxic
 7. Threat
 8. Identity_Attack
 9. Toxicity_Score
10. Comment_Length
11. Word_Count
12. Platform
13. Topic
14. Moderation_Priority
15. Language


### 1.8 Data types

`df.dtypes` shows the type of each column:

- **int64** – whole numbers (for example 0/1 label flags or word counts)
- **float64** – decimal numbers (for example a toxicity score)
- **object / str** – text

`df.info()` also shows the number of non-null values per column and the memory usage.

In [8]:
df.dtypes.to_frame(name="dtype")

,dtype
Comment_ID,str
Comment_Text,str
Toxicity_Label,str
Clean,int64
Toxic,int64
Severe_Toxic,int64
Threat,int64
Identity_Attack,int64
Toxicity_Score,float64
Comment_Length,int64


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Comment_ID           30000 non-null  str    
 1   Comment_Text         30000 non-null  str    
 2   Toxicity_Label       30000 non-null  str    
 3   Clean                30000 non-null  int64  
 4   Toxic                30000 non-null  int64  
 5   Severe_Toxic         30000 non-null  int64  
 6   Threat               30000 non-null  int64  
 7   Identity_Attack      30000 non-null  int64  
 8   Toxicity_Score       30000 non-null  float64
 9   Comment_Length       30000 non-null  int64  
 10  Word_Count           30000 non-null  int64  
 11  Platform             30000 non-null  str    
 12  Topic                30000 non-null  str    
 13  Moderation_Priority  30000 non-null  str    
 14  Language             30000 non-null  str    
dtypes: float64(1), int64(7), str(7)
memory usage: 3

### 1.9 Statistical summary of numerical columns

`df.describe()` calculates, for each numerical column, the count, mean, standard deviation,
minimum, quartiles (25%, 50% = median, 75%), and maximum.

For the 0/1 label columns, the **mean** is the proportion of comments that have that label.

In [10]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Clean,30000.0,0.619300,0.485567,0.0,0.000,1.000,1.000,1.00
Toxic,30000.0,0.380700,0.485567,0.0,0.000,0.000,1.000,1.00
Severe_Toxic,30000.0,0.060033,0.237553,0.0,0.000,0.000,0.000,1.00
Threat,30000.0,0.048633,0.215104,0.0,0.000,0.000,0.000,1.00
Identity_Attack,30000.0,0.048933,0.215732,0.0,0.000,0.000,0.000,1.00
Toxicity_Score,30000.0,0.324542,0.302914,0.0,0.079,0.161,0.624,0.95
Comment_Length,30000.0,56.357633,6.962596,39.0,52.000,58.000,61.000,71.00
Word_Count,30000.0,8.340633,0.972094,6.0,8.000,8.000,9.000,11.00


### 1.10 Missing values

`df.isnull().sum()` counts empty values in each column. Missing comment text or missing labels
would have to be handled before training.

In [11]:
missing = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().mean() * 100).round(2),
})
print("Total missing values in the dataset:", int(missing["missing_count"].sum()))
missing

Total missing values in the dataset: 0


,missing_count,missing_percent
Comment_ID,0,0.0
Comment_Text,0,0.0
Toxicity_Label,0,0.0
Clean,0,0.0
Toxic,0,0.0
Severe_Toxic,0,0.0
Threat,0,0.0
Identity_Attack,0,0.0
Toxicity_Score,0,0.0
Comment_Length,0,0.0


### 1.11 Duplicate records

We check duplicates at three levels:

1. **Fully duplicated rows** – every column is identical.
2. **Duplicated `Comment_ID` values** – IDs should be unique.
3. **Duplicated `Comment_Text` values** – the same comment text appearing more than once
   (this matters for NLP because repeated text can leak between training and test sets).

In [12]:
duplicate_summary = pd.Series({
    "Fully duplicated rows": df.duplicated().sum(),
    "Duplicated Comment_ID values": df["Comment_ID"].duplicated().sum(),
    "Duplicated Comment_Text values": df["Comment_Text"].duplicated().sum(),
}, name="count")
duplicate_summary.to_frame()

,count
Fully duplicated rows,0
Duplicated Comment_ID values,0
Duplicated Comment_Text values,0


### 1.12 Quick look at the text and categorical columns

To prepare for the next task, we check how many unique values each text/categorical column has.
A column with very few unique values (such as a label) is categorical; a column where almost
every value is unique (such as the comment text) is free text.

In [13]:
text_columns = df.select_dtypes(exclude="number").columns
df[text_columns].nunique().to_frame(name="unique_values")

,unique_values
Comment_ID,30000
Comment_Text,30000
Toxicity_Label,5
Platform,5
Topic,6
Moderation_Priority,3
Language,1


### Task 1 Summary

The outputs above show:

- the dataset's real size and its 15 columns;
- **`Comment_Text`** is the free-text column we will use as model input;
- **`Toxicity_Label`** is a single categorical column, and the binary columns `Clean`, `Toxic`,
  `Severe_Toxic`, `Threat`, and `Identity_Attack` hold 0/1 label flags; both are analysed in Task 2;
- the missing-value and duplicate counts, which tell us how much cleaning is needed.

Next, Task 2 cleans the comment text and analyses the target labels.